# Local / Anaconda Runner (v4)
Same unified pipeline as Colab; data root defaults to `data/cleaned_data` (override with `--data-dir`). Run `python training/run.py --env` first and paste it into any bug report.


In [ ]:
# v4 bootstrap: mount Drive, install pinned deps, verify cleaned data
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO = Path("/content/minor-project") if IN_COLAB else Path.cwd()
if not (REPO / "training" / "run.py").exists():
    # adjust to your Drive copy of the repo
    REPO = Path("/content/drive/MyDrive/object-detection/minor-project")
os.chdir(REPO)
print("repo:", REPO)

# single installer entry (idempotent, W8.1)
subprocess.check_call([sys.executable, "install.py"])

# environment report - paste this into any bug report (W8.5)
subprocess.check_call([sys.executable, "training/run.py", "--env"])

# verify cleaned data; if missing, run the cleaner first:
#   python data_cleaning/clean_build_v4.py --help
DATA = Path("/content/drive/MyDrive/object-detection/cleaned_data") if IN_COLAB else REPO / "data" / "cleaned_data"
for ds in ("codrone", "visdrone", "dota"):
    ok = (DATA / ds / "train" / "images").is_dir()
    print(f"{ds:<10}: {'ready' if ok else 'MISSING - run data_cleaning/clean_build_v4.py'}")


## Smoke test


In [ ]:
# CI smoke check: all 23 models on the generated mock dataset (no GPU/data needed)
!python training/run.py --models all --datasets mock     --steps stage,train,eval,summary --test


## Train


In [ ]:
# Single model, tuned defaults apply automatically - no flags needed
!python training/run.py --models lsknet --datasets codrone


In [ ]:
# Full benchmark: every model x every cleaned dataset, tuned defaults (S2.6),
# background staging, failure isolation, auto-resume, Drive mirroring.
!python training/run.py --models all --datasets all --mode full
